# Milestone 001 — Accessible does not imply used

A compact recurrent state can preserve task-relevant information even when the model's own frozen downstream consumer makes poor use of it.

This notebook explains the distinction with a runnable synthetic example. The public version deliberately omits the private corpus, checkpoints, active experiment queue, and end-to-end reproduction details.

## The scientific question

Suppose a frozen hidden state `h` comes from a model whose native next-step predictor performs poorly on a particular distinction. There are at least two very different explanations:

1. **Representation failure:** the relevant information is no longer present in a form a small downstream consumer can recover.
2. **Utilization failure:** the information remains readily accessible, but the native consumer does not exploit it well.

Those hypotheses suggest different interventions. If the representation failed, we may need to change training or architecture. If utilization failed, a tiny readout repair may be enough.


> **Sticky note — frozen state:** Frozen means we do not change the representation-producing model while testing the readout. This prevents a better result from being explained by the representation itself adapting. [Deeper reference →](../reference/glossary.md#frozen-representation)

> **Sticky note — readout / consumer:** The consumer is the downstream function that converts a hidden representation into predictions. A probe is a diagnostic readout trained specifically to ask what information is accessible from that representation. [Deeper reference →](../reference/representation-and-readout.md)


## Why an affine-softmax probe?

An **affine-softmax probe** computes logits `z = W h + b` and converts them to class probabilities with softmax.

> **Sticky note — affine:** An affine map is a linear transformation plus a bias. It can rotate, rescale, mix, and shift coordinates, but it cannot invent nonlinear feature interactions. [Deeper reference →](../reference/glossary.md#affine-map)

> **Sticky note — softmax:** Softmax turns arbitrary logits into a probability distribution. It does not make the decision boundary nonlinear in `h`; for a fixed pair of classes, the boundary is still determined by an affine comparison of logits. [Deeper reference →](../reference/glossary.md#softmax)

So success with this probe says something deliberately modest but useful: the distinction is available to a very small, linearly parameterized consumer.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score

rng = np.random.default_rng(7)
n, d = 6000, 32
y = rng.integers(0, 2, size=n)
h = rng.normal(size=(n, d))

task_direction = rng.normal(size=d)
task_direction /= np.linalg.norm(task_direction)
h += (2 * y[:, None] - 1) * 0.65 * task_direction

X_train, X_test, y_train, y_test = train_test_split(h, y, test_size=0.35, random_state=11, stratify=y)
probe = LogisticRegression(max_iter=2000).fit(X_train, y_train)
p_probe = probe.predict_proba(X_test)

native_direction = rng.normal(size=d)
native_direction -= native_direction.dot(task_direction) * task_direction
native_direction /= np.linalg.norm(native_direction)
native_logits = X_test @ native_direction
p1 = 1 / (1 + np.exp(-native_logits))
p_native = np.c_[1 - p1, p1]
constant = np.full((len(y_test), 2), 0.5)

print(f'constant baseline NLL: {log_loss(y_test, constant):.3f}')
print(f'native consumer NLL:   {log_loss(y_test, p_native):.3f}')
print(f'affine probe NLL:      {log_loss(y_test, p_probe):.3f}')
print(f'probe accuracy:         {accuracy_score(y_test, probe.predict(X_test)):.3f}')


The important pattern is not the exact synthetic number. It is the ordering: **small probe ≫ native consumer ≈ weak baseline**. That pattern means we should not immediately blame the representation.

### A real milestone behind this example

In the private experiment that motivated this note, a frozen compact state supported a strong affine-softmax branch probe on held-out natural-source examples, while the native frozen consumer captured only a small fraction of that accessible gain. Selectivity controls and an alternate coordinate treatment reproduced the qualitative conclusion.

> **Validated milestone:** task-relevant branch information is strongly accessible to a small affine readout, but substantially underused by the native frozen consumer.

The next active experiment is intentionally not reproduced here.


## Why the controls matter

A successful probe is not automatically interesting. A sufficiently flexible classifier can sometimes exploit dataset artifacts, leakage, or accidental memorization.

Useful controls ask whether the result survives attacks on those explanations: shuffle labels; shuffle representations relative to examples; compare with matched random features; preserve the exact train/evaluation boundary; and replicate in a coordinate system that removes an obvious geometric artifact.

> **Sticky note — selectivity:** A probe is selective when it succeeds on the intended structure while appropriately failing controls where that structure has been destroyed. [Deeper reference →](../reference/glossary.md#selectivity)

> **Sticky note — held-out:** Held-out data was not used to fit the probe or choose its hyperparameters. It tests generalization rather than fit quality. [Deeper reference →](../reference/glossary.md#held-out-evaluation)


In [ ]:
shuffled = rng.permutation(y_train)
control_probe = LogisticRegression(max_iter=2000).fit(X_train, shuffled)
print(f'shuffled-label control NLL: {log_loss(y_test, control_probe.predict_proba(X_test)):.3f}')
print(f'shuffled-label control accuracy: {accuracy_score(y_test, control_probe.predict(X_test)):.3f}')


## What this does not show

A successful affine probe does **not** prove that the native model causally uses the probed information; that all task-relevant information is present; that the representation is globally sufficient; that an unrestricted nonlinear decoder could not recover still more information; or that the probe result will translate into better full next-token/next-byte predictions.

> **Sticky note — accessibility vs causal use:** A probe asks, “Can this information be decoded?” A causal intervention asks, “Does the model's behavior actually depend on it?” Those are not the same question. [Deeper reference →](../reference/experimental-reasoning.md#accessibility-versus-use)


## Understanding questions

**Why affine?**  
Because it tests accessibility with a deliberately small linear readout family before invoking nonlinear decoder capacity.

**Does softmax make the probe nonlinear in the representation?**  
No. It normalizes affine logits into probabilities; pairwise decision boundaries remain affine.

**Why freeze the representation?**  
So improved predictions cannot be attributed to changing the representation itself.

**Why isn't a successful probe proof that the model uses the information?**  
Because a separately trained decoder can exploit information that the native consumer ignores.

**Why might a probe just be memorizing?**  
If capacity is large relative to the dataset, or train/evaluation dependence leaks structure, it can fit accidental associations rather than generalizable task signal.

**What does a label-shuffle control test?**  
Whether the probe pipeline can manufacture apparent performance after the target relationship has been destroyed.

**What result would support representation failure instead?**  
Failure of preregistered bounded probe families, with machinery-positive and selectivity controls passing, supports a bounded non-recoverability claim—not unrestricted information absence.

**What should you test next after accessible-but-underused?**  
Hold representation and data fixed, then test a ladder of increasingly expressive small consumers to locate the minimum readout repair that materially improves the actual prediction task.


## Research-engineering takeaway

The value of the experiment is not merely that one classifier scored better. It changed the intervention target.

Before the probe, poor task behavior was compatible with representation loss. After a strong, controlled frozen-state probe, the more economical hypothesis is that at least part of the problem lies downstream in utilization. That turns a broad architecture question into a much smaller readout question.

That transition—from ambiguous observation to a discriminating experiment to a narrower next hypothesis—is the central research-engineering pattern these notes will document.
